# Convert Allen V1 (core_nll_7) to libsonata-compatible circuit

The original core_nll_7 config uses `${configdir}` (BMTK convention) which
libsonata/bluepysnap cannot parse. This notebook creates a standard SONATA
circuit config with absolute paths and writes it to disk.

**Output:** `../../../obi-output/nest_core_nll_7/circuit/`

In [ ]:
import json
from pathlib import Path

CIRCUIT_BASE = Path("/Users/james/Documents/obi/code/obi-main/core_nll_7")
NETWORK_DIR = CIRCUIT_BASE / "network"
OUTPUT_DIR = Path("../../../obi-output/nest_core_nll_7/circuit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Create node_sets.json and circuit_config.json

In [ ]:
nd = str(NETWORK_DIR)

# Create node_sets.json with an "All" entry for the v1 population
node_sets = {"All": {"population": "v1"}}
node_sets_path = OUTPUT_DIR / "node_sets.json"
print(node_sets_path.absolute())
with node_sets_path.open("w", encoding="utf-8") as f:
    json.dump(node_sets, f, indent=2)

circuit_config = {
    "manifest": {"$BASE_DIR": "."},
    "node_sets_file": str(node_sets_path.resolve().absolute()),
    "networks": {
        "nodes": [
            {"nodes_file": f"{nd}/v1_nodes.h5", "populations": {"v1": {"type": "point_process"}}},
            {"nodes_file": f"{nd}/lgn_nodes.h5", "populations": {"lgn": {"type": "virtual"}}},
            {"nodes_file": f"{nd}/bkg_nodes.h5", "populations": {"bkg": {"type": "virtual"}}},
        ],
        "edges": [
            {
                "edges_file": f"{nd}/v1_v1_edges_bio_trained.h5",
                "populations": {"v1_to_v1": {"type": "chemical"}},
            },
            {
                "edges_file": f"{nd}/lgn_v1_edges.h5",
                "populations": {"lgn_to_v1": {"type": "chemical"}},
            },
            {
                "edges_file": f"{nd}/bkg_v1_edges_bio_trained.h5",
                "populations": {"bkg_to_v1": {"type": "chemical"}},
            },
        ],
    },
}

circuit_config_path = OUTPUT_DIR / "circuit_config.json"
with circuit_config_path.open("w", encoding="utf-8") as f:
    json.dump(circuit_config, f, indent=2)

print(f"Circuit config written to {circuit_config_path}")
print(f"Node sets written to {node_sets_path}")

In [ ]:
# Verify the output
with circuit_config_path.open() as f:
    written = json.load(f)

print(f"Node populations: {[list(n['populations'].keys())[0] for n in written['networks']['nodes']]}")
print(f"Edge populations: {[list(e['populations'].keys())[0] for e in written['networks']['edges']]}")